In [ ]:
from importlib import reload

from Trainers.trainer_classifier import ClassifierTrainer, ClassifierTrainingConfig
from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
from Models.model_classifier import Classifier
import Models.model_evaluator as model_evaluator
import Datasets.dataset_request as dataset_request
from datasets import load_dataset
import torch
import gc


dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset

In [ ]:
# 1.) Create dataset
dataset_name: str = "Anthropic/hh-rlhf"
dataset = load_dataset(dataset_name)
new_ds = RequestDataset.from_raw(dataset, "Qwen/Qwen3-0.6B")
torch.save(new_ds["prompts"], "human_requests_hh-rlhf.pt")

dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
dataset.truncate(0, 1024)
# 2.) Add reward and policy and value
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")


# 3.) PPO training 
config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy",
    batch_size = 16
)
trainer = PolicyPPOTrainer(policy, reward_model, value, dataset, config)
trainer.train()

# 4.) Delete reward and value

del reward_model
del value


gc.collect()
torch.cuda.empty_cache()


# 5.) generate answers from policy.
policy.generate_new_dataset(dataset)

# 6.) Move policy to cpu.
policy.model.to("cpu")


# 7.) Create evaluator and evaluate policy
PrometheusEvaluator = model_evaluator.PrometheusEvaluator
evaluator = PrometheusEvaluator()
print(evaluator.evaluate(policy))

# 8.) Delete evaluator
evaluator.model.to("cpu")
del evaluator

gc.collect()
torch.cuda.empty_cache()


# Repeat for stronger judge
